# Stage 2 - Instruction Fine-Tuning (SFT)

**Project:** PrepMind - GenAI / Agentic AI Learning Assistant
**Base model:** Stage-1 non-instruction adapter (loaded from the Hugging Face Hub) on top
of `Qwen2.5-1.5B-Instruct` (4-bit, Unsloth)
**Goal:** Teach the domain-adapted model to actually answer questions - given a GenAI /
Agentic AI question, produce a clear, correct, interview-ready answer, instead of just
continuing text.

Run on a Colab **T4 GPU** runtime. This notebook pushes its output adapter to the
**Hugging Face Hub** so Stage 3 can load it in a fresh Colab session.


## 1. Install dependencies

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub


## Hugging Face Hub setup

Same `HF_USERNAME` you used in `non_instruction_finetuning.ipynb` - this notebook reads
that notebook's pushed adapter by repo id, and pushes its own adapter for Stage 3 to read.

In [ ]:
HF_USERNAME = "your-hf-username"  # <-- set this once, same value in all 3 notebooks
NON_INSTRUCTION_REPO = f"{HF_USERNAME}/prepmind-non-instruction-adapter"
SFT_REPO = f"{HF_USERNAME}/prepmind-sft-adapter"

from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass("Enter your Hugging Face access token (needs write access): ")
login(token=token)


## 2. Clone the project repo

In [ ]:
REPO_URL = "https://github.com/Abhishek15016/prepmind.git"
!git clone $REPO_URL project
%cd project


## 3. Load the base model

Loads directly from the Stage-1 adapter's **Hugging Face Hub repo** (pushed at the end of
`non_instruction_finetuning.ipynb`), so this works even in a brand-new Colab session with
no local files carried over. Set `USE_STAGE1_ADAPTER = False` to skip Stage 1 and start
from the plain base model instead.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
USE_STAGE1_ADAPTER = True

base_model_name = NON_INSTRUCTION_REPO if USE_STAGE1_ADAPTER else "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)


## 4. Apply LoRA adapters for instruction tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()


## 5. Load and format the instruction dataset

`data/instruction_dataset.jsonl` stores each example as a `messages` list
(`system` / `user` / `assistant`), which we render through Qwen2.5's official chat
template so the model sees exactly the token format it will be served with at
inference time.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen2.5")

raw_ds = load_dataset("json", data_files="data/instruction_dataset.jsonl", split="train")
print(raw_ds)
print(raw_ds[0])


In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

formatted_ds = raw_ds.map(format_example, remove_columns=raw_ds.column_names)
formatted_ds = formatted_ds.train_test_split(test_size=0.1, seed=42)
print(formatted_ds["train"][0]["text"][:600])


## 6. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/instruction_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_ds["train"],
    eval_dataset=formatted_ds["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
    packing=False,
)

trainer_stats = trainer.train()


## 7. Save the SFT adapter locally, then push to the Hugging Face Hub

In [ ]:
model.save_pretrained("outputs/sft_adapter")
tokenizer.save_pretrained("outputs/sft_adapter")
print("Saved SFT LoRA adapter locally to outputs/sft_adapter")

model.push_to_hub(SFT_REPO, private=True)
tokenizer.push_to_hub(SFT_REPO, private=True)
print(f"Pushed adapter to https://huggingface.co/{SFT_REPO}")
print("Use this exact repo id as SFT_ADAPTER_PATH in dpo_alignment.ipynb")


## 8. Run inference on the 10 evaluation questions

These are the same 10 questions used in `reports/base_model_evaluation.md` (base model,
before any fine-tuning) so we can build a fair before/after comparison in
`reports/sft_model_comparison.md`.

In [ ]:
EVAL_QUESTIONS = [
    "What is the difference between LoRA and QLoRA?",
    "Why does self-attention need positional encoding?",
    "What is Retrieval-Augmented Generation and why does it reduce hallucination?",
    "Explain the difference between an LLM agent and a simple prompted LLM call.",
    "What is DPO, and how is it different from RLHF with PPO?",
    "Why do we chunk documents before embedding them for RAG?",
    "What is the KV cache and why does it speed up LLM inference?",
    "What is Model Context Protocol (MCP) and what problem does it solve?",
    "How would you evaluate a RAG system end-to-end?",
    "What is catastrophic forgetting, and why is LoRA less prone to it than full fine-tuning?",
]

FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": "You are a friendly expert tutor in Generative AI and Agentic AI. Explain concepts in depth with intuitive analogies, practical production insight, and interview-ready takeaways, in a clear and engaging style."},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    text = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text.strip()

for q in EVAL_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)


## Notes

- Copy the printed Q/A pairs above into `reports/sft_model_comparison.md` alongside the
  base model's answers to build the before/after comparison table.
- Stage 3 (`dpo_alignment.ipynb`) loads the `{HF_USERNAME}/prepmind-sft-adapter` repo you
  just pushed and aligns it further with the preference dataset.
